# Data Loading

This notebook builds a knowledge graph from SEC 10-K filing data. You will load a sample filing, split it into chunks, and store everything in Neo4j with relationships that preserve document structure.

**Learning Objectives:**
- Understand the Document-Chunk graph pattern
- Connect to Neo4j from a Jupyter notebook
- Create Document and Chunk nodes
- Link chunks to their source document and to each other

## The Document-Chunk Pattern

GraphRAG applications split source documents into smaller pieces called **chunks** for two reasons: LLMs have limited context windows, and smaller chunks produce more precise retrieval results.

The graph structure preserves both the source and the reading order:

```
(:Document) <-[:FROM_DOCUMENT]- (:Chunk) -[:NEXT_CHUNK]-> (:Chunk)
```

- **FROM_DOCUMENT** links every chunk back to its source, so the retriever can report provenance.
- **NEXT_CHUNK** connects consecutive chunks, so a retrieval query can grab surrounding context to give the LLM a wider window.

In [ ]:
%pip install neo4j python-dotenv -q

In [ ]:
import os

from lib.data_utils import get_embedder
from neo4j import GraphDatabase
from dotenv import load_dotenv

# Load configuration
load_dotenv('../CONFIG.txt')

NEO4J_URI = os.getenv('NEO4J_URI')
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Connected to Neo4j successfully!')

## Clear Existing Data

Start with a clean graph by removing any nodes and relationships from previous runs.

In [ ]:
def clear_graph(driver):
    """Remove all nodes and relationships from the database."""
    with driver.session() as session:
        result = session.run("""
            MATCH (n)
            DETACH DELETE n
            RETURN count(n) as deleted
        """)
        count = result.single()['deleted']
        print(f'Deleted {count} nodes')

clear_graph(driver)

## Load Financial Filing Data

Load the sample 10-K filing text from `financial_data.txt`. This file contains excerpts from Apple's annual report covering products, services, risk factors, and financial performance.

In [ ]:
with open('financial_data.txt', 'r') as f:
    filing_text = f.read()

print(f'Loaded {len(filing_text)} characters')
print(f'\nPreview (first 300 chars):\n{filing_text[:300]}...')

## Create Document Node

Create a Document node to represent the source filing. This node stores metadata about the filing and serves as the anchor for all chunks extracted from it.

In [ ]:
def create_document(driver, name: str, source: str):
    """Create a Document node with the given name and source."""
    with driver.session() as session:
        result = session.run("""
            MERGE (d:Document {name: $name})
            SET d.source = $source
            RETURN elementId(d) as doc_id
        """, name=name, source=source)
        doc_id = result.single()['doc_id']
        print(f'Created Document node: {name}')
        return doc_id

doc_id = create_document(driver, 'Apple Inc 10-K 2024', 'SEC EDGAR')
print(f'Document ID: {doc_id}')

## Split Text into Chunks

Split the filing text into smaller chunks for embedding and retrieval. Each chunk is 500 characters with 50 characters of overlap between consecutive chunks. The overlap ensures that sentences split across chunk boundaries still appear in at least one complete chunk.

In [ ]:
def split_into_chunks(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    """Split text into chunks of chunk_size characters with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

chunks = split_into_chunks(filing_text)
print(f'Split into {len(chunks)} chunks\n')
for i, chunk in enumerate(chunks):
    print(f'Chunk {i}: {len(chunk)} chars')
    print(f'  {chunk[:80]}...\n')

## Create Chunk Nodes and Link to Document

Create a Chunk node for each piece of text and connect it to the Document node with a `FROM_DOCUMENT` relationship. Then link consecutive chunks with `NEXT_CHUNK` relationships to preserve reading order.

In [ ]:
def create_chunks(driver, doc_id: str, chunks: list[str]):
    """Create Chunk nodes linked to a Document via FROM_DOCUMENT."""
    chunk_data = [{"index": i, "text": t} for i, t in enumerate(chunks)]
    with driver.session() as session:
        session.run("""
            MATCH (d:Document) WHERE elementId(d) = $doc_id
            UNWIND $chunks AS chunk
            MERGE (c:Chunk {index: chunk.index})
            SET c.text = chunk.text
            MERGE (c)-[:FROM_DOCUMENT]->(d)
        """, doc_id=doc_id, chunks=chunk_data)
        print(f'Created {len(chunks)} Chunk nodes with FROM_DOCUMENT relationships')


def link_chunks(driver, num_chunks: int):
    """Create NEXT_CHUNK relationships between consecutive chunks."""
    pairs = [{"idx1": i, "idx2": i + 1} for i in range(num_chunks - 1)]
    with driver.session() as session:
        session.run("""
            UNWIND $pairs AS pair
            MATCH (c1:Chunk {index: pair.idx1})
            MATCH (c2:Chunk {index: pair.idx2})
            MERGE (c1)-[:NEXT_CHUNK]->(c2)
        """, pairs=pairs)
        print(f'Created {num_chunks - 1} NEXT_CHUNK relationships')


create_chunks(driver, doc_id, chunks)
link_chunks(driver, len(chunks))

## Verify Graph Structure

Query the graph to confirm that all nodes and relationships were created correctly.

In [ ]:
def show_graph_structure(driver):
    """Display a summary of the graph structure."""
    with driver.session() as session:
        # Count nodes by label
        node_result = session.run("""
            MATCH (n)
            WITH labels(n)[0] AS label, n
            WHERE label IS NOT NULL
            RETURN label, count(n) AS count
            ORDER BY label
        """)
        print('=== Node Counts ===')
        for record in node_result:
            print(f'  {record["label"]}: {record["count"]}')

        # Count relationships by type
        rel_result = session.run("""
            MATCH ()-[r]->()
            RETURN type(r) AS type, count(r) AS count
            ORDER BY type
        """)
        print('\n=== Relationship Counts ===')
        for record in rel_result:
            print(f'  {record["type"]}: {record["count"]}')

        # Show chunk chain
        chain_result = session.run("""
            MATCH (c:Chunk)
            OPTIONAL MATCH (c)-[:NEXT_CHUNK]->(next:Chunk)
            RETURN c.index AS idx, next.index AS next_idx,
                   left(c.text, 60) AS preview
            ORDER BY c.index
        """)
        print('\n=== Chunk Chain ===')
        for record in chain_result:
            next_str = f' -> Chunk {record["next_idx"]}' if record['next_idx'] is not None else ' (end)'
            print(f'  Chunk {record["idx"]}: "{record["preview"]}..."{next_str}')

show_graph_structure(driver)

## Summary

You built a Document-Chunk graph from SEC 10-K filing text:

1. **Document node** stores the filing metadata (name and source)
2. **Chunk nodes** hold individual text segments from the filing
3. **FROM_DOCUMENT** relationships link every chunk to its source document
4. **NEXT_CHUNK** relationships preserve the original reading order

This graph structure is the foundation for everything that follows. In the next notebook, you will generate vector embeddings for each chunk so the graph supports semantic search.

---

**Next:** [Embeddings](02_embeddings.ipynb)

In [ ]:
# Cleanup
driver.close()
print('Connection closed.')